# National Energy Demand Forecasting

You're an ML engineer at a national power grid operator. The grid needs accurate
short-term demand forecasts to balance supply, schedule maintenance, and prevent
blackouts. Under-predict and you risk blackouts. Over-predict and you waste
energy and money. Getting it right matters.

## Loading and exploring the data

In [ ]:
%pip install seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df = pd.read_csv("./energy_demand.csv", parse_dates=["date"])
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Regions: {df['region'].unique()}")
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for region in df["region"].unique():
    region_data = df[df["region"] == region]
    ax.plot(region_data["date"], region_data["demand_mw"], alpha=0.5, label=region, linewidth=0.8)
ax.set_xlabel("Date")
ax.set_ylabel("Demand (MW)")
ax.set_title("Daily energy demand by region")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
day_demand = df.groupby(df["date"].dt.dayofweek)["demand_mw"].mean()
axes[0].bar(day_names, day_demand.values)
axes[0].set_title("Average demand by day of week")
axes[0].set_ylabel("Demand (MW)")

weather_demand = df.groupby("weather_type")["demand_mw"].mean().sort_values()
axes[1].barh(weather_demand.index, weather_demand.values)
axes[1].set_title("Average demand by weather type")
axes[1].set_xlabel("Demand (MW)")

plt.tight_layout()
plt.show()

## Feature engineering for time series

Time series data has temporal structure — yesterday's demand tells you something
about today's. We need to create features that capture this.

In [ ]:
# Parse date components
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["day_of_year"] = df["date"].dt.dayofyear
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

Lag features let the model see recent demand values. A 1-day lag captures
short-term momentum; a 7-day lag captures weekly patterns. Rolling averages
smooth out noise and reveal trends.

In [ ]:
# Sort by region and date to ensure correct ordering
df = df.sort_values(["region", "date"]).reset_index(drop=True)

# Lag features: what was demand yesterday? Last week?
df["demand_lag_1"] = df.groupby("region")["demand_mw"].shift(1)
df["demand_lag_7"] = df.groupby("region")["demand_mw"].shift(7)

# Rolling averages
df["demand_rolling_7"] = df.groupby("region")["demand_mw"].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
df["demand_rolling_30"] = df.groupby("region")["demand_mw"].transform(
    lambda x: x.rolling(30, min_periods=1).mean()
)

# Drop rows with NaN from lagging
df = df.dropna()
print(f"Shape after feature engineering: {df.shape}")

**Why random splits are wrong for time series.** If we randomly split the data,
the training set might contain data from December 2022 while the test set
contains data from January 2022. The model would be "predicting" the past using
the future — that's data leakage. We must split temporally: train on earlier
data, test on later data.

In [ ]:
# Temporal split: train on 2021, test on 2022
train_mask = df["date"] < "2022-01-01"
test_mask = df["date"] >= "2022-01-01"

print(f"Training: {train_mask.sum()} rows ({df[train_mask]['date'].min()} to {df[train_mask]['date'].max()})")
print(f"Test:     {test_mask.sum()} rows ({df[test_mask]['date'].min()} to {df[test_mask]['date'].max()})")

## Categorical features

`region` and `weather_type` are strings, not numbers. The model can't work with
strings directly. We need to encode them.

**One-hot encoding:** Create a binary column for each category. "North" becomes
`region_North=1, region_South=0, region_Midlands=0`.

In [ ]:
# One-hot encode categorical features
df_encoded = pd.get_dummies(df, columns=["region", "weather_type"], drop_first=True)
print(f"Columns before encoding: {len(df.columns)}")
print(f"Columns after encoding: {len(df_encoded.columns)}")
print(f"\nNew columns: {[c for c in df_encoded.columns if c not in df.columns]}")

The dataset now has more columns — each categorical value became its own binary
feature. `drop_first=True` avoids redundancy (if it's not North or Midlands, it
must be South).

In [ ]:
# Define features (drop target and date)
feature_cols = [c for c in df_encoded.columns if c not in ["date", "demand_mw"]]
X_train = df_encoded[train_mask][feature_cols]
X_test = df_encoded[test_mask][feature_cols]
y_train = df_encoded[train_mask]["demand_mw"]
y_test = df_encoded[test_mask]["demand_mw"]

print(f"Features: {len(feature_cols)}")
print(f"Training: {X_train.shape}")
print(f"Test: {X_test.shape}")

## Gradient boosting

Tree-based models handle mixed feature types naturally — they don't need feature
scaling. Gradient boosting builds many small decision trees sequentially, each
one correcting the errors of the previous ones. scikit-learn's
`GradientBoostingRegressor` implements this. In production, libraries like
XGBoost and LightGBM are popular alternatives.

In [ ]:
# Train a GradientBoostingRegressor
# Use n_estimators=200, max_depth=5, random_state=42
model = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
model.fit(X_train, y_train)
print("Model trained!")

## Evaluation

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"MAE:  {mae:.0f} MW")
print(f"RMSE: {rmse:.0f} MW")

In [ ]:
# Plot predictions vs actuals over time
test_dates = df_encoded[test_mask]["date"]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_dates, y_test.values, alpha=0.7, label="Actual", linewidth=0.8)
ax.plot(test_dates, y_pred, alpha=0.7, label="Predicted", linewidth=0.8)
ax.set_xlabel("Date")
ax.set_ylabel("Demand (MW)")
ax.set_title("Predictions vs Actuals (2022)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols)
importance.nlargest(10).plot(kind="barh", figsize=(8, 5))
plt.title("Top 10 feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## Asymmetric error

Not all errors are equal. Under-predicting demand leads to power shortages and
potential blackouts. Over-predicting leads to wasted capacity and higher costs.
The grid operator would rather over-predict slightly than risk a blackout.

In [ ]:
# Standard regression minimises squared error (symmetric)
# Quantile regression can target a specific quantile
# alpha=0.9 means the model targets the 90th percentile — it will tend to over-predict
model_upper = GradientBoostingRegressor(
    loss="quantile", alpha=0.9, n_estimators=200, max_depth=5, random_state=42
)
model_upper.fit(X_train, y_train)
y_pred_upper = model_upper.predict(X_test)

# Compare
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_dates[:60], y_test.values[:60], label="Actual", linewidth=1.5)
ax.plot(test_dates[:60], y_pred[:60], label="Mean prediction", linewidth=1, alpha=0.8)
ax.plot(test_dates[:60], y_pred_upper[:60], label="90th percentile", linewidth=1, alpha=0.8, linestyle="--")
ax.set_xlabel("Date")
ax.set_ylabel("Demand (MW)")
ax.set_title("Mean vs conservative (upper quantile) predictions")
ax.legend()
plt.tight_layout()
plt.show()

The upper quantile prediction deliberately over-estimates. A grid operator might
prefer this — the cost of over-provisioning is much lower than the cost of a
blackout.

## Drift

A model trained on 2021-2022 data reflects the world as it was then. But the
world changes: new power plants come online, energy efficiency improves,
populations shift. If the world changes and the model doesn't, its predictions
will silently degrade. This is called **drift**.

In [ ]:
drift_df = pd.read_csv("./energy_demand_drift.csv", parse_dates=["date"])
print(f"Drift data: {drift_df.shape}")
print(f"Date range: {drift_df['date'].min()} to {drift_df['date'].max()}")

In [ ]:
# Apply the same feature engineering
drift_df["day_of_week"] = drift_df["date"].dt.dayofweek
drift_df["month"] = drift_df["date"].dt.month
drift_df["day_of_year"] = drift_df["date"].dt.dayofyear
drift_df["is_weekend"] = (drift_df["day_of_week"] >= 5).astype(int)

drift_df = drift_df.sort_values(["region", "date"]).reset_index(drop=True)
drift_df["demand_lag_1"] = drift_df.groupby("region")["demand_mw"].shift(1)
drift_df["demand_lag_7"] = drift_df.groupby("region")["demand_mw"].shift(7)
drift_df["demand_rolling_7"] = drift_df.groupby("region")["demand_mw"].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
drift_df["demand_rolling_30"] = drift_df.groupby("region")["demand_mw"].transform(
    lambda x: x.rolling(30, min_periods=1).mean()
)
drift_df = drift_df.dropna()

drift_encoded = pd.get_dummies(drift_df, columns=["region", "weather_type"], drop_first=True)

# Ensure same columns as training data (handle missing dummies)
for col in feature_cols:
    if col not in drift_encoded.columns:
        drift_encoded[col] = 0
X_drift = drift_encoded[feature_cols]
y_drift = drift_encoded["demand_mw"]

drift_pred = model.predict(X_drift)
drift_mae = mean_absolute_error(y_drift, drift_pred)
print(f"Training period MAE: {mae:.0f} MW")
print(f"Drift period MAE:    {drift_mae:.0f} MW")

In [ ]:
# Compare demand distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["demand_mw"], bins=30, alpha=0.7, label="2021-2022")
axes[0].hist(drift_df["demand_mw"], bins=30, alpha=0.7, label="2023-Q1")
axes[0].set_title("Demand distribution shift")
axes[0].legend()

axes[1].hist(df["temperature"], bins=30, alpha=0.7, label="2021-2022")
axes[1].hist(drift_df["temperature"], bins=30, alpha=0.7, label="2023-Q1")
axes[1].set_title("Temperature distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## Monitoring

How would you detect this in production? You wouldn't have labelled demand data
immediately — the actual demand figures come after the fact. But you could:

1. Track prediction error over rolling time windows (e.g. weekly MAE)
2. Monitor feature distributions for shift (e.g. is the temperature distribution
   this month different from the training data?)
3. Set alert thresholds (e.g. "alert if weekly MAE exceeds 2x the training MAE")

### Discussion

- How often should this model be retrained?
- What would trigger an emergency retraining?
- Who should be alerted when drift is detected?